# SignalForge Week 11 TRL ORPO Colab Notebook

This notebook avoids Unsloth and uses a simpler, more stable stack for Colab:

- `transformers`
- `trl`
- `peft`
- `bitsandbytes`

Use this notebook if the Unsloth notebook keeps failing with package mismatch or restart-loop issues.


In [ ]:
# Run this on a fresh Colab runtime with a T4 GPU.
!pip install --upgrade pip setuptools wheel
!pip install --no-cache-dir \
  "torch==2.6.0" \
  "transformers==4.51.3" \
  "trl==0.18.2" \
  "peft==0.15.2" \
  "accelerate==1.6.0" \
  "bitsandbytes==0.45.5" \
  "datasets==3.4.1" \
  sentencepiece


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
from pathlib import Path
import json
import subprocess

REPO_URL = 'https://github.com/nebiyuephrata/SignalForge.git'
REPO_BRANCH = 'main'
REPO_ROOT = Path('/content/SignalForge')

if not REPO_ROOT.exists():
    subprocess.run(['git', 'clone', '--branch', REPO_BRANCH, REPO_URL, str(REPO_ROOT)], check=True)
else:
    print(f'Reusing existing repo at {REPO_ROOT}')

TRAIN_EXPORT = REPO_ROOT / 'training_data' / 'unsloth' / 'preferences_train.jsonl'
DEV_EXPORT = REPO_ROOT / 'training_data' / 'unsloth' / 'preferences_dev.jsonl'
HELD_OUT_EXPORT = REPO_ROOT / 'training_data' / 'unsloth' / 'preferences_held_out.jsonl'
OUTPUT_DIR = REPO_ROOT / 'training' / 'colab_trl_orpo_outputs'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print({'repo_root': str(REPO_ROOT), 'output_dir': str(OUTPUT_DIR)})


In [ ]:
%cd /content/SignalForge
!python training/export_unsloth_datasets.py


In [ ]:
from datasets import Dataset

def read_jsonl(path):
    with open(path) as f:
        return [json.loads(line) for line in f if line.strip()]

train_rows = read_jsonl(TRAIN_EXPORT)
dev_rows = read_jsonl(DEV_EXPORT)

print({'train_rows': len(train_rows), 'dev_rows': len(dev_rows)})
print(train_rows[0]['task_id'], train_rows[0]['dimension'])


In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

MODEL_NAME = 'Qwen/Qwen2.5-1.5B-Instruct'
MAX_LENGTH = 1536
MAX_PROMPT_LENGTH = 1024

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'right'

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    torch_dtype=torch.float16,
    device_map='auto',
)

model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=16,
    lora_alpha=16,
    lora_dropout=0.05,
    bias='none',
    task_type='CAUSAL_LM',
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'],
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()


## Optional SFT Warm Start

Leave this off for the first run.


In [ ]:
RUN_SFT_WARMSTART = False

if RUN_SFT_WARMSTART:
    from trl import SFTTrainer, SFTConfig

    warm_train = Dataset.from_list([{'text': row['sft_text']} for row in train_rows])
    warm_dev = Dataset.from_list([{'text': row['sft_text']} for row in dev_rows])

    sft_trainer = SFTTrainer(
        model=model,
        tokenizer=tokenizer,
        train_dataset=warm_train,
        eval_dataset=warm_dev,
        dataset_text_field='text',
        max_seq_length=MAX_LENGTH,
        args=SFTConfig(
            output_dir=str(OUTPUT_DIR / 'sft_warmstart'),
            per_device_train_batch_size=2,
            gradient_accumulation_steps=4,
            max_steps=80,
            learning_rate=2e-4,
            logging_steps=5,
            eval_strategy='steps',
            eval_steps=20,
            save_steps=40,
            fp16=True,
            report_to='none',
        ),
    )
    sft_trainer.train()
else:
    print('Skipping SFT warm start.')


In [ ]:
from trl import ORPOConfig, ORPOTrainer

orpo_train = Dataset.from_list([
    {
        'prompt': row['prompt'],
        'chosen': row['chosen'],
        'rejected': row['rejected'],
        'task_id': row['task_id'],
        'dimension': row['dimension'],
    }
    for row in train_rows
])

orpo_dev = Dataset.from_list([
    {
        'prompt': row['prompt'],
        'chosen': row['chosen'],
        'rejected': row['rejected'],
        'task_id': row['task_id'],
        'dimension': row['dimension'],
    }
    for row in dev_rows
])

orpo_args = ORPOConfig(
    output_dir=str(OUTPUT_DIR / 'orpo_run'),
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=1e-5,
    num_train_epochs=2,
    logging_steps=5,
    save_steps=50,
    eval_strategy='steps',
    eval_steps=25,
    max_length=MAX_LENGTH,
    max_prompt_length=MAX_PROMPT_LENGTH,
    beta=0.1,
    fp16=True,
    report_to='none',
    remove_unused_columns=False,
)

trainer = ORPOTrainer(
    model=model,
    args=orpo_args,
    train_dataset=orpo_train,
    eval_dataset=orpo_dev,
    processing_class=tokenizer,
)

trainer.train()


In [ ]:
adapter_dir = OUTPUT_DIR / 'orpo_adapter'
trainer.save_model(str(adapter_dir))
tokenizer.save_pretrained(str(adapter_dir))
print({'adapter_dir': str(adapter_dir)})


## Evaluation Discipline

- Tune on `train` and `dev` only.
- Keep `held_out` sealed until the recipe is fixed.
- First success criterion: the run finishes and dev metrics improve without overconfident tone drift.
